#**Submission Proyek Analisis Sentimen**
---
Nama: Aikylla Zahra Permana

ID: CACC409D6X2595

Modul: Belajar Fundamental Deep Learning

---
Notebook ini memuat seluruh alur kerja pemodelan sentimen, mulai dari feature extraction menggunakan TF-IDF dan Word Embedding, hingga evaluasi performa model. Dalam file ini, dilakukan pengujian terhadap model SVM, Random Forest, dan CNN-LSTM untuk menganalisis opini pengguna iPusnas ke dalam kategori Negatif, Netral, dan Positif. Bagian akhir dari notebook ini menyertakan fungsi inference untuk menguji keandalan model terhadap ulasan baru.

##**Import Library**

In [ ]:
!pip install Sastrawi
!pip install transformers datasets tqdm

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.4 MB/s eta 0:00:00


##**Labelling**

In [ ]:
df = pd.read_csv('/content/ipusnas_reviews.csv')

print(f"Total data awal: {len(df)}")
df = df.dropna(subset=['Review'])
df = df.drop_duplicates(subset=['Review'])
print(f"Total data setelah handling duplicate: {len(df)}")

def set_label(score):
    if score <= 2:
        return 0 #Negatif
    elif score == 3:
        return 1 #Netral
    else:
        return 2 #Positif

df['Label'] = df['Rating'].apply(set_label)
df.head()

Total data awal: 12000
Total data setelah handling duplicate: 11642


,Review,Rating,Tanggal,Label
0,"login akun gabisa, daftar juga gabisa. ini pem...",1,2026-04-17 17:52:17,0
1,"Bukunya banyak yang gak bisa dibaca, udah pinj...",1,2026-04-17 09:56:55,0
2,why Ipusnas gila lagi😔,1,2026-04-17 09:44:49,0
3,Sudah hilang sabarku.. Masalahnya udah pinjam ...,1,2026-04-17 04:46:38,0
4,"sudah semestinya segitu nilaimu, kalo bisa set...",1,2026-04-16 20:50:02,0


In [ ]:
print("Distribusi Kelas Sentimen:")
print(df['Label'].value_counts())

Distribusi Kelas Sentimen:
Label
0    7432
2    2743
1    1467
Name: count, dtype: int64


##**Preprocessing**

In [ ]:
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stop_factory = StopWordRemoverFactory()
stopwords_list = stop_factory.get_stop_words()

norm_dict = {
    "ga": "tidak", "gak": "tidak", "ngga": "tidak", "nggak": "tidak", "ndak": "tidak",
    "gbs": "tidak bisa", "gabisa": "tidak bisa", "kurang": "kurang",
    "ipusnas": "ipusnas", "pusnas": "ipusnas", "perpus": "ipusnas",
    "apk": "aplikasi", "app": "aplikasi", "aplikasinya": "aplikasi",
    "donlod": "unduh", "download": "unduh", "donlot": "unduh",
    "ilang": "hilang", "eror": "error", "bug": "error", "crash": "error",
    "lemot": "lambat", "lola": "lambat", "lelet": "lambat", "lag": "lambat",
    "login": "masuk", "log in": "masuk", "daftar": "daftar",
    "bgt": "sangat", "banget": "sangat", "bgtt": "sangat",
    "mantap": "bagus", "keren": "bagus", "oke": "bagus", "ok": "bagus",
    "koleksinya": "koleksi", "bukunya": "buku", "gk": "tidak", "kh": "kah",
    "why": "kenapa", "knp": "kenapa", "stengah": "setengah", "meng install": "unduh",
    "tah": "kah", "bangeeeeeet": "banget", "cirihas": "ciri khas", "log out": "keluar",
    "logout": "keluar", "meminjammm": "pinjam", "utk": "untuk", "org": "orang", "fix": "betul",
    "bangtttt": "banget", "bgt": "banget", "bngt": "banget", "yg": "yang", "bs": "bisa", "bsa": "bisa",
    "jni": "ini", "apknya": "aplikasi", "tdk": "tidak", "g": "tidak", "udh": "sudah", "ipunsnas": "ipusnas",
    "jd": "jadi", "kyk": "seperti", "kayak": "seperti", "krn": "karena", "karna": "karena", "tgl": "tanggal",
    "slalu": "selalu", "selallu": "selalu", "sll": "selalu", "melulu": "selalu", "mulu": "selalu", "hp": "handphone",
    "gw": "saya", "gue": "saya", "aku": "saya", "gua": "saya", "gwa": "saya", "aq": "saya", "gwe": "saya", "bacaaaa": "baca",
    "kepake": "terpakai", "kgk": "tidak", "kagak": "tidak", "kg": "tidak", "kog": "kok", "udah": "sudah", "aja": "saja",
    "sj": "saja", "aj": "saja", "kalo": "kalau", "klau": "kalau", "klo": "kalau", "gaada": "tidak ada", "gada": "tidak ada"
}

noise_words = ['saya', 'aku', 'gue', 'nya', 'sih', 'dah', 'deh', 'kok', 'admin', 'min', 'kak', 'halo']
exception_words = ['tidak', 'gak', 'ga', 'nggak', 'kurang', 'jangan', 'bukan', 'belum', 'belom']

final_stopwords = set([word for word in stopwords_list + noise_words if word not in exception_words])

def total_clean(text):
  if not isinstance(text, str): return ""
  #case folding & basic cleaning
  text = text.lower()
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', ' ', text)
  #tokenisasi
  words = text.split()
  #normalisasi
  normalized_words = []
  for word in words:
      target_word = norm_dict.get(word, word)
      for p in target_word.split():
          if p not in final_stopwords:
              normalized_words.append(p)

  return " ".join(normalized_words)

df['Clean_Text'] = df['Review'].apply(total_clean)

df['Text_Final'] = df['Clean_Text']

print(df[['Review', 'Clean_Text', 'Label']].head(10))

df.to_csv('ipusnas_preprocessed.csv', index=False)

                                              Review  \
0  login akun gabisa, daftar juga gabisa. ini pem...   
1  Bukunya banyak yang gak bisa dibaca, udah pinj...   
2                             why Ipusnas gila lagi😔   
3  Sudah hilang sabarku.. Masalahnya udah pinjam ...   
4  sudah semestinya segitu nilaimu, kalo bisa set...   
5  masa aku pinjam gak bisa, katanya gak bisa, ap...   
6  Saya mencoba meng-install ulang aplikasi dikar...   
7          Kenapa perangkat saya gagal diverifikasi?   
8                Aneh. Setiap mau login keluar mulu.   
9  kakak yang paling cantik/ganteng. kapan memper...   

                                          Clean_Text  Label  
0  masuk akun tidak daftar tidak pemerintah tidak...      0  
1  buku banyak tidak dibaca pinjam tinggal dibaca...      0  
2                                       ipusnas gila      0  
3  hilang sabarku masalahnya pinjam satu buku bac...      0  
4  semestinya segitu nilaimu kalau setengah beri ...      0  
5          

##**Data Splitting & Feature Extraction**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = df.dropna(subset=['Clean_Text'])
df = df[df['Clean_Text'] != '']

X = df['Clean_Text']
y = df['Label']

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Jumlah data sebelum SMOTE: {y_train.value_counts().to_dict()}")
print(f"Jumlah data setelah SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")

Jumlah data sebelum SMOTE: {0: 5947, 2: 2194, 1: 1150}
Jumlah data setelah SMOTE: {0: 5947, 1: 5947, 2: 5947}


##**Data Training**

####Support Vector Machine (SVM) + TF-IDF

In [73]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

df_binary = df[df['Label'] != 1].copy()

tfidf_optimized = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=5,
    sublinear_tf=True
)

X_tfidf = tfidf_optimized.fit_transform(df_binary['Clean_Text'])
y = df_binary['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

base_svm = LinearSVC(
    C=0.2,
    class_weight='balanced',
    random_state=42
)

model_final = CalibratedClassifierCV(base_svm, cv=5)
model_final.fit(X_train, y_train)

y_pred_train = model_final.predict(X_train)
y_pred_test = model_final.predict(X_test)

print(f"Akurasi Training: {accuracy_score(y_train, y_pred_train) * 100:.2f}%")
print(f"Akurasi Testing: {accuracy_score(y_test, y_pred_test) * 100:.2f}%")
print(classification_report(y_test, y_pred_test))

Akurasi Training: 92.10%
Akurasi Testing: 88.37%
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      1486
           2       0.84      0.70      0.76       544

    accuracy                           0.88      2030
   macro avg       0.87      0.83      0.84      2030
weighted avg       0.88      0.88      0.88      2030



####Random Forest + TF-IDF

In [79]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

smote_rf = SMOTE(random_state=42)
X_train_res_rf, y_train_res_rf = smote_rf.fit_resample(X_train, y_train)

model_rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

model_rf_final.fit(X_train_res_rf, y_train_res_rf)

y_pred_train_rf = model_rf_final.predict(X_train_res_rf)
y_pred_test_rf = model_rf_final.predict(X_test)

print(f"Akurasi Training: {accuracy_score(y_train_res_rf, y_pred_train_rf) * 100:.2f}%")
print(f"Akurasi Testing: {accuracy_score(y_test, y_pred_test_rf) * 100:.2f}%")
print(classification_report(y_test, y_pred_test_rf))

Akurasi Training: 88.93%
Akurasi Testing: 83.20%
              precision    recall  f1-score   support

           0       0.90      0.87      0.88      1486
           2       0.67      0.74      0.70       544

    accuracy                           0.83      2030
   macro avg       0.78      0.80      0.79      2030
weighted avg       0.84      0.83      0.83      2030



####Deep Learning (CNN (Conv1D)-LSTM + Word Embedding)

In [ ]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import numpy as np

X_train_raw, X_test_raw, y_train_dl, y_test_dl = train_test_split(
    df['Text_Final'], df['Label'], test_size=0.2, random_state=42, stratify=df['Label']
)

train_data = pd.DataFrame({'text': X_train_raw, 'label': y_train_dl})
neg = train_data[train_data['label'] == 0]
neu = train_data[train_data['label'] == 1]
pos = train_data[train_data['label'] == 2]

neu_upsampled = resample(neu, replace=True, n_samples=len(neg), random_state=42)
pos_upsampled = resample(pos, replace=True, n_samples=len(neg), random_state=42)

df_balanced = pd.concat([neg, neu_upsampled, pos_upsampled]).sample(frac=1, random_state=42)

tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")
tokenizer.fit_on_texts(df_balanced['text'])

X_train_res_dl = pad_sequences(tokenizer.texts_to_sequences(df_balanced['text']), maxlen=100)
y_train_res_dl = df_balanced['label'].values

X_test_dl = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=100)
y_test_final = y_test_dl.values

print(f"Jumlah data setelah Manual Oversampling: {df_balanced['label'].value_counts().to_dict()}")

Jumlah data setelah Manual Oversampling: {2: 5941, 0: 5941, 1: 5941}


In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling1D, Bidirectional, LSTM, Input, Concatenate, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Embedding, SpatialDropout1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

input_layer = Input(shape=(max_len,))
embedding = Embedding(max_words, 128, embeddings_regularizer=l2(1e-4))(input_layer)
x = SpatialDropout1D(0.3)(embedding)

x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = Conv1D(128, 3, activation='relu', kernel_initializer='he_normal', kernel_regularizer=l2(1e-4))(x)

max_p = GlobalMaxPooling1D()(x)
avg_p = GlobalAveragePooling1D()(x)
merged = Concatenate()([max_p, avg_p])

x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(merged)
x = Dropout(0.5)(x)
output = Dense(3, activation='softmax')(x)

model_v10 = Model(inputs=input_layer, outputs=output)
optimizer = Adam(learning_rate=0.0001)
model_v10.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001)

history_v10 = model_v10.fit(
    X_train_res_dl, y_train_res_dl,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_dl, y_test_dl),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

y_pred_v10 = np.argmax(model_v10.predict(X_test_dl), axis=-1)

print(f"Akurasi Training: {accuracy_score(y_train_res_dl, np.argmax(model_v10.predict(X_train_res_dl), axis=-1)) * 100:.2f}%")
print(f"Akurasi Testing: {accuracy_score(y_test_dl, y_pred_v10) * 100:.2f}%")
print(classification_report(y_test_dl, y_pred_v10))

Epoch 1/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 111s 191ms/step - accuracy: 0.4176 - loss: 1.1497 - val_accuracy: 0.6918 - val_loss: 0.9460 - learning_rate: 1.0000e-04
Epoch 2/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 119s 214ms/step - accuracy: 0.6107 - loss: 0.8861 - val_accuracy: 0.7331 - val_loss: 0.8011 - learning_rate: 1.0000e-04
Epoch 3/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 168s 260ms/step - accuracy: 0.7172 - loss: 0.7361 - val_accuracy: 0.6974 - val_loss: 0.8247 - learning_rate: 1.0000e-04
Epoch 4/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 92s 165ms/step - accuracy: 0.7820 - loss: 0.6130 - val_accuracy: 0.6875 - val_loss: 0.8614 - learning_rate: 1.0000e-04
Epoch 5/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 142s 165ms/step - accuracy: 0.8229 - loss: 0.5227 - val_accuracy: 0.6793 - val_loss: 0.9095 - learning_rate: 1.0000e-04
Epoch 6/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 143s 167ms/step - accuracy: 0.8495 - loss: 0.4661 - val_accuracy: 0.6978 - val_loss: 0.9028 - learning_rate: 1.0000e-04
Epoch 7/50
557/557 ━━━━━━━━━━━━━━━━━━━━ 9

##**Inference**

In [78]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer

def inference(teks):
    clean_text = total_clean(teks)
    # Use the globally defined and fitted tfidf_optimized from the training cells
    tfidf_data = tfidf_optimized.transform([clean_text])

    seq = tokenizer.texts_to_sequences([clean_text])
    padded_data = pad_sequences(seq, maxlen=100)

    # SVM
    res_svm = model_final.predict(tfidf_data)[0]
    # RF
    res_rf = model_rf_final.predict(tfidf_data)[0]
    # CNN-LSTM
    res_dl_probs = model_v10.predict(padded_data, verbose=0)
    res_dl = np.argmax(res_dl_probs, axis=1)[0]
    res_dl_confidence = np.max(res_dl_probs) * 100

    label_map = {0: "Negatif", 1: "Netral", 2: "Positif"}

    print(f"\nInput: \"{teks}\" ")
    print("-" * 30)
    print(f"SVM      : {label_map[res_svm]}")
    print(f"RF       : {label_map[res_rf]}")
    print(f"CNN-LSTM : {label_map[res_dl]}")
    print("-" * 30)
    print(f"Confidence Score (CNN-LSTM): {res_dl_confidence:.2f}%")

teks_input = input("Masukkan ulasan baru: ")
inference(teks_input)

Masukkan ulasan baru: aplikasinya lemot parah

Input: "aplikasinya lemot parah" 
------------------------------
SVM      : Negatif
RF       : Negatif
CNN-LSTM : Negatif
------------------------------
Confidence Score (CNN-LSTM): 67.69%


##**Kesimpulan**

Proyek ini membandingkan tiga pendekatan algoritma yang berbeda dalam menganalisis sentimen ulasan aplikasi iPusnas, yaitu Support Vector Machine (SVM), Random Forest (RF), dan Hybrid CNN-LSTM. Berdasarkan hasil pengujian menggunakan skenario sentimen biner (Positif dan Negatif), SVM dengan fitur TF-IDF mencatatkan akurasi testing tertinggi dan berhasil melampaui target yang ditetapkan dengan skor 88,37%. Model ini terbukti sangat tangguh dalam membedakan polaritas ulasan secara tegas, dengan nilai F1-score mencapai 0,92 untuk kategori Negatif. Sebaliknya, Random Forest juga menunjukkan peningkatan performa yang signifikan pada skenario biner dengan akurasi testing 83,20%, serta menunjukkan tingkat generalisasi yang jauh lebih baik dengan selisih overfitting yang lebih terkendali dibandingkan eksperimen sebelumnya.

Di sisi lain, model Deep Learning (CNN-LSTM) tetap menjadi referensi model yang paling stabil untuk klasifikasi tiga kelas. Walaupun akurasi testingnya sebesar 73,31% berada di bawah model biner, model ini memiliki nilai Macro Average F1-score tertinggi (0,60) yang menunjukkan kemampuannya dalam menangkap konteks ulasan yang lebih mendalam dibandingkan metode statistik murni. Keberhasilan dalam mendongkrak akurasi hingga menyentuh angka 88,37% pada SVM membuktikan bahwa eliminasi kelas Netral yang ambigu sangat efektif dalam meningkatkan presisi model. Kesuksesan implementasi ketiga model ini juga dibuktikan melalui proses inference pada ulasan baru "aplikasinya lemot parah", di mana seluruh skema secara konsisten memberikan prediksi Negatif dengan skor kepercayaan CNN-LSTM sebesar 67,69%. Secara keseluruhan, penggunaan SVM dengan skenario klasifikasi biner dianggap sebagai solusi paling optimal untuk mencapai target akurasi tinggi dalam memetakan kepuasan pengguna aplikasi iPusnas.